In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
from shapely.geometry import MultiPoint, Point, Polygon, shape
import json
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


from feems.utils import prepare_graph_inputs
from feems import SpatialGraph, Viz
from feems.cross_validation import run_cv

UP = Path(r'/Users/ram/Library/CloudStorage/Box-Box/Ram_Ximena_Nicole/Indp Research Phillipine Languages/data')
GRID_RES = "grid_100.shp"
DISC_R   = 1.25
USE_IDF  = True                          # IDF-weight columns (grammar's only supported mode; see Cell 7)

In [ ]:
META_PATH = UP / "grammar_feems_meta.json"

# This file is written by the last cell of a full run -- it's a record of exactly
# which settings produced whatever edgew_grammar.csv / grammar_surface_raster.csv
# currently sit in your data folder. Read it here, at the TOP of the notebook, so you
# see last run's config before you accidentally re-run with different settings and
# silently invalidate the exported CSVs (e.g. changing COAST_BUF but not re-exporting).

if META_PATH.exists():
    with open(META_PATH) as f:
        prev_meta = json.load(f)
    print("=== metadata from previous run ===")
    for k, v in prev_meta.items():
        print(f"  {k}: {v}")
    print()
    print("If you change GRID_RES, COAST_BUF, or SCALE_SNPS below and don't")
    print("re-run the cells below, the exported CSVs will silently reflect the OLD")
    print("settings shown above, not your new ones. lamb_cv in particular does NOT")
    print("carry over across a habitat or grid change -- always re-run the CV cell after.")
else:
    prev_meta = None
    print("No previous run metadata found -- this will be the first full run.")
    print("grammar_feems_meta.json will be written at the end.")

In [ ]:
# --- REPRODUCIBILITY NOTES ---
# 1. GRID_RES, COAST_BUF, SCALE_SNPS below are the ONLY knobs that should change
#    between runs. Changing any of them invalidates lamb_cv from a prior run --
#    re-run the cross-validation cell every time one of these changes.
# 2. The habitat cell (real coastline, buffer=COAST_BUF) and lamb_cv are coupled:
#    a different habitat has a different edge count, so the CV curve and its
#    argmin will differ even at the "same" lambda_grid.
# 3. grammar_feems_meta.json is the audit trail -- if a figure looks different from
#    a previous version, diff this file first before re-checking code.
# 4. This notebook mirrors python/phoneme_feems.ipynb, adapted for GRAMBANK's
#    multi-level categorical features (see the OHE + per-value IDF weighting in
#    the genotype-matrix cell below, which replaces phoneme's simple per-column
#    IDF multiply -- GRAMBANK features are not all binary like phoneme's are).

In [ ]:
# Loading Data

gram = pd.read_csv(UP / "GRAMBANKdf_full.csv")
freq = pd.read_csv(UP / "gramfeature_freq.csv")

GB_ALL = [c for c in gram.columns if c.startswith("GB")]
print(gram.shape, "|", len(GB_ALL), "GRAMBANK feature cols")
print(gram.Language_Type.value_counts())

In [ ]:
# Subset to the Philippine Languages
ph = gram.loc[gram.Language_Type == "Philippine Language"].reset_index(drop=True)

coord = ph[["Longitude", "Latitude"]].to_numpy()      # FEEMS wants (lon, lat) -- same convention as phoneme_feems.ipynb
n_ph = coord.shape[0]
print("n Philippine languages:", n_ph)
print("lon", coord[:,0].min().round(2), "..", coord[:,0].max().round(2),
      "| lat", coord[:,1].min().round(2), "..", coord[:,1].max().round(2))
print("duplicate coords:", len(coord) - len(np.unique(coord, axis=0)))

In [ ]:
# One-hot encode categorical (>2-level) columns -- mirrors
# [1]_GRAMMAR_cosine_similarity.R's OHE step exactly, so the FEEMS genotype
# matrix and the R cosine-similarity matrix are built from the same features.
n_levels = {c: gram[c].dropna().nunique() for c in GB_ALL}
binary_cols      = [c for c in GB_ALL if n_levels[c] <= 2]
categorical_cols = [c for c in GB_ALL if n_levels[c] > 2]
print(f"Binary (kept):       {len(binary_cols)} cols")
print(f"Categorical (OHE'd): {len(categorical_cols)} cols")

ph_ohe = pd.get_dummies(ph, columns=categorical_cols, prefix=categorical_cols)
ohe_cols = [c for c in ph_ohe.columns
            if any(c.startswith(f"{cc}_") for cc in categorical_cols)]
GB_ALL_OHE = binary_cols + ohe_cols
print(f"Feature matrix: {len(binary_cols)} binary + {len(ohe_cols)} OHE cols = {len(GB_ALL_OHE)} total features")

In [ ]:
# Feature filtering (zero-variance columns contribute nothing to the surface)

X_all  = ph_ohe[GB_ALL_OHE].to_numpy(dtype=float)
col_n  = X_all.sum(axis=0)
n_lang = X_all.shape[0]

keep     = (col_n > 0) & (col_n < n_lang)          # drop zero-variance columns
GB_KEEP  = [c for c, k in zip(GB_ALL_OHE, keep) if k]
OHE_KEEP = [c for c in GB_KEEP if c in ohe_cols]

print(f"total {len(GB_ALL_OHE)} | all-zero {(col_n==0).sum()} | all-one {(col_n==n_lang).sum()} | retained {keep.sum()}")

Xc = X_all[:, keep] - X_all[:, keep].mean(axis=0)
print("rank(centred X):", int((np.linalg.svd(Xc, compute_uv=False) > 1e-10).sum()), "of", keep.sum())

In [ ]:
# Pseudo genotype matrix for GRAMBANK data -- IDF-weighted
#
# Unlike phoneme (all-binary features -> geno = presence*2 * per-column IDF),
# GRAMBANK has multi-level categorical features, so the weighting mirrors
# [1]_GRAMMAR_cosine_similarity.R's calculate_weighted_cosine_similarity()
# exactly: gramfeature_freq.csv is keyed by (feature, value), not by feature
# alone.
#   - Binary (non-OHE) columns: weighted_value = IDF of whichever value (0/1)
#     was actually observed for that language (IDF differs between 0 and 1).
#   - OHE dummy columns (e.g. "GB065_1"): weighted_value = dummy * IDF(base
#     feature, level) -- the standard one-hot + IDF scheme, since
#     gramfeature_freq has no row for the dummy name itself, only for the
#     original (feature, raw value) pair.
long = ph_ohe[["language"] + GB_KEEP].melt(id_vars="language", var_name="feature", value_name="value")

def base_and_level(col):
    if col in OHE_KEEP:
        base, level = col.rsplit("_", 1)
        return base, float(level)
    return col, None

bl = long["feature"].map(base_and_level)
long["is_ohe"]        = long["feature"].isin(OHE_KEEP)
long["lookup_feature"] = [b for b, _ in bl]
long["lookup_value"]   = [lvl if lvl is not None else v
                          for (_, lvl), v in zip(bl, long["value"])]

freq_idx = freq.set_index(["feature", "value"])["IDF"]
long["IDF"] = long.set_index(["lookup_feature", "lookup_value"]).index.map(freq_idx)
assert not long["IDF"].isna().any(), "unmatched (feature, value) pairs in gramfeature_freq.csv"

long["weighted_value"] = np.where(long["is_ohe"], long["value"] * long["IDF"], long["IDF"])

geno = (long.pivot(index="language", columns="feature", values="weighted_value")
            .loc[ph_ohe["language"], GB_KEEP]
            .to_numpy(dtype=float))
SCALE_SNPS = False   # weighting is already IDF-substituted above; do not also Patterson-scale

print(geno.shape, "| range", geno.min().round(3), geno.max().round(3), "| NaN:", np.isnan(geno).any())

In [ ]:
from shapely.ops import unary_union

with open(UP / "countries.geojson") as f:
    countries = json.load(f)

ph_feat = next(f for f in countries["features"] if f["properties"].get("name") == "Philippines")
ph_geom = shape(ph_feat["geometry"])

# Unlike phoneme (all 58 Philippine languages sit within the Philippines proper),
# grammar's Philippine-language set also includes several Sabah/Sama-Bajau
# languages (Rungus, Sabah Malay, Tambunan Dusun, Tatana, Timugon Murut, West
# Coast Bajau) that sit in Malaysian NW Borneo -- outside the Philippines-only
# habitat, which would otherwise strand them from the node mesh. Extend the
# habitat with Malaysia's Borneo-side landmass (Peninsular Malaysia, lon < 108,
# is irrelevant here and would needlessly balloon the habitat), mirroring how
# [3]_GRAMMAR_network_distance.R already unions "Philippines" + "Malaysia" for
# its own land/sea mask.
my_feat  = next(f for f in countries["features"] if f["properties"].get("name") == "Malaysia")
my_geom  = shape(my_feat["geometry"])
my_borneo = unary_union([g for g in my_geom.geoms if g.bounds[0] >= 108])

COAST_BUF = 0.8   # minimum buffer that merges the archipelago into one connected piece --
                  # do not lower without re-checking; 0.7 leaves Sulu Archipelago isolated
habitat = unary_union([ph_geom, my_borneo]).buffer(COAST_BUF).simplify(0.1)
assert habitat.geom_type == "Polygon", f"still {len(habitat.geoms)} pieces -- raise COAST_BUF"
outer = np.array(habitat.exterior.coords)
print("outer vertices:", len(outer))

In [ ]:
import feems
import os
print(os.path.join(os.path.dirname(feems.__file__), "data", GRID_RES))

In [ ]:
import feems
import os

grid_path = os.path.join(os.path.dirname(feems.__file__), "data", GRID_RES)
assert os.path.exists(grid_path), f"grid shapefile not found at {grid_path}"

outer, edges, grid, ipmap = prepare_graph_inputs(
    coord=coord,
    ggrid=grid_path,
    translated=False,
    buffer=0,
    outer=outer,       # from the habitat cell -- the union-of-discs polygon
)

# --- connectivity: resistance distance is undefined across disconnected components ---
G = nx.Graph()
G.add_nodes_from(range(grid.shape[0]))
G.add_edges_from([(a - 1, b - 1) for a, b in edges])   # edges are 1-indexed
n_components = nx.number_connected_components(G)

print(f"grid nodes: {grid.shape[0]}  |  edges: {edges.shape[0]}  |  components: {n_components}")
assert n_components == 1, (
    f"habitat fragmented into {n_components} disconnected pieces -- raise DISC_R above"
)

# --- coverage: every language must land inside the node mesh, not just the polygon ---
node_hull = MultiPoint([tuple(g) for g in grid]).convex_hull
stranded = [i for i, c in enumerate(coord) if not node_hull.contains(Point(c))]
print(f"stranded languages: {len(stranded)}")
assert len(stranded) == 0, f"{len(stranded)} language(s) fall outside the node mesh: {stranded}"

# --- occupancy sanity check ---
occ_idx = np.unique(ipmap)
occ_counts = np.array([np.sum(ipmap == i) for i in occ_idx])
print(f"occupied demes: {len(occ_idx)} of {grid.shape[0]}  "
      f"({(grid.shape[0]-len(occ_idx))/grid.shape[0]*100:.0f}% unobserved)")
print(f"languages per occupied deme -- max: {occ_counts.max()}, median: {int(np.median(occ_counts))}")

In [ ]:
# SpatialGraph + occupancy check

sp_graph = SpatialGraph(geno, coord, grid, edges, scale_snps=SCALE_SNPS)

occ = np.array([sp_graph.nodes[n]["n_samples"] for n in range(len(sp_graph.nodes))])
occ = occ[occ > 0]
print("observed demes:", sp_graph.n_observed_nodes, "| max languages/deme:", occ.max(), "| median:", np.median(occ))
print("languages per occupied deme:", np.sort(occ)[::-1])
print("singleton demes:", (occ == 1).sum(), "of", len(occ))

In [ ]:
# Patching run_cv bugged feems

import numpy as np
from copy import deepcopy
import feems.cross_validation as cv_mod
from feems.objective import Objective, comp_mats
from feems.spatial_graph import query_node_attributes

def predict_snps_patched(sp_graph, sp_graph_train, sp_graph_test):
    obj = Objective(sp_graph)
    obj.sp_graph.comp_graph_laplacian(sp_graph_train.w)
    fit_cov, _, _ = comp_mats(obj)

    n_snps = sp_graph.n_snps
    if sp_graph.scale_snps:
        mu_f = np.sqrt(sp_graph.mu * (1 - sp_graph.mu))   # original behaviour, untouched
    else:
        mu_f = np.ones(n_snps)                             # frequencies already in natural units

    frequencies_ns = sp_graph.frequencies * mu_f
    mu0 = frequencies_ns.mean(axis=0) / 2
    mu_frequencies = 2 * mu0 / mu_f
    frequencies = deepcopy(sp_graph.frequencies)

    ids = np.empty(sp_graph.n_observed_nodes, dtype=bool)
    permuted_idx = query_node_attributes(sp_graph, "permuted_idx")
    permuted_idx_train = query_node_attributes(sp_graph_train, "permuted_idx")
    for i, idx in enumerate(permuted_idx[: sp_graph.n_observed_nodes]):
        ids[i] = idx in permuted_idx_train[: sp_graph_train.n_observed_nodes]
    train_ids = np.argwhere(ids).reshape(-1)
    test_ids  = np.argwhere(~ids).reshape(-1)

    cov_te_tr = fit_cov[np.ix_(test_ids, train_ids)]
    cov_tr_tr = fit_cov[np.ix_(train_ids, train_ids)]
    pred_frequencies = mu_frequencies + cov_te_tr @ np.linalg.solve(
        cov_tr_tr, frequencies[train_ids, :] - mu_frequencies
    )
    l2_err = np.linalg.norm(
        pred_frequencies * mu_f - frequencies[test_ids] * mu_f
    )
    return pred_frequencies, l2_err

cv_mod.predict_snps = predict_snps_patched   # redirect run_cv's internal call

In [ ]:
# Unlike phoneme (optimum lambda ~0.30, well inside a 1e-6..1e2 grid), grammar's
# weaker/near-absent spatial signal (consistent with [5]_GRAMMAR_MMRR.R's null
# geography/phylogeny result) pushes the CV optimum much higher (~67) -- close
# to phoneme's grid boundary. Widened to 1e-6..1e4 (same log-density) so the
# true minimum sits safely in the interior rather than at the search edge.
lamb_grid = np.geomspace(1e-6, 1e4, 25)[::-1]

cv_err = run_cv(sp_graph, lamb_grid, n_folds=sp_graph.n_observed_nodes, factr=1e10)
mean_cv_err = np.mean(cv_err, axis=0)
lamb_cv = float(lamb_grid[np.argmin(mean_cv_err)])

fig, ax = plt.subplots(figsize=(5,3), dpi=120)
ax.plot(np.log10(lamb_grid), mean_cv_err, "o-")
ax.axvline(np.log10(lamb_cv), ls="--", c="r")
ax.set_xlabel(r"$\log_{10}\lambda$"); ax.set_ylabel("LOO CV error")
plt.show()

print("lamb_cv =", lamb_cv)
if np.argmin(mean_cv_err) in (0, len(lamb_grid)-1):
    print("WARNING: minimum at grid boundary -- widen lamb_grid")

In [ ]:
sp_graph.fit(lamb=lamb_cv, optimize_q=None)

print("edges:", len(sp_graph.w))
print("log10(w/wbar):  min %.3f  max %.3f  sd %.3f" % (
    np.log10(sp_graph.w / sp_graph.w.mean()).min(),
    np.log10(sp_graph.w / sp_graph.w.mean()).max(),
    np.log10(sp_graph.w / sp_graph.w.mean()).std()))

In [ ]:
projection = ccrs.EquidistantConic(central_longitude=122.0, central_latitude=13.0)

fig = plt.figure(figsize=(7,9), dpi=200)
ax  = fig.add_subplot(1,1,1, projection=projection)
v = Viz(ax, sp_graph, projection=projection, edge_width=0.6,
        edge_alpha=1, edge_zorder=100, sample_pt_size=12,
        obs_node_size=8, sample_pt_color="black", cbar_font_size=8)
v.draw_map()
v.draw_edges(use_weights=True)
v.draw_obs_nodes(use_ids=False)
v.draw_edge_colorbar()
plt.show()

In [ ]:

n_nodes = sp_graph.node_pos.shape[0]
edge_arr = np.array(sp_graph.edges)          # (n_edges, 2), 0-indexed
log_w = np.log10(sp_graph.w / sp_graph.w.mean())

node_val = np.zeros(n_nodes)
node_cnt = np.zeros(n_nodes)
for k in range(len(edge_arr)):
    i, j = edge_arr[k]
    node_val[i] += log_w[k]; node_cnt[i] += 1
    node_val[j] += log_w[k]; node_cnt[j] += 1

node_val = np.divide(node_val, node_cnt, out=np.full(n_nodes, np.nan), where=node_cnt > 0)
print("isolated nodes (no edges, will be NaN):", np.isnan(node_val).sum())

In [ ]:
triang = mtri.Triangulation(sp_graph.node_pos[:,0], sp_graph.node_pos[:,1])
interp = mtri.LinearTriInterpolator(triang, node_val)

RES = 300  # raster resolution; raise for a smoother/finer export
lon_g = np.linspace(outer[:,0].min(), outer[:,0].max(), RES)
lat_g = np.linspace(outer[:,1].min(), outer[:,1].max(), int(RES*4/3))
LON, LAT = np.meshgrid(lon_g, lat_g)
Z = interp(LON, LAT)

poly = Polygon(outer)   # reuse the exact outer polygon from the habitat cell -- same habitat, same graph
mask = np.array([[poly.contains(Point(LON[i,j], LAT[i,j])) for j in range(LON.shape[1])]
                  for i in range(LON.shape[0])])
Z = np.ma.array(Z, mask=~mask | np.ma.getmaskarray(Z))

print("valid raster fraction:", (~np.ma.getmaskarray(Z)).mean().round(3))

In [ ]:
# EEMS-convention colormap: orange = low conductance (barrier), cyan = high
feems_cmap = LinearSegmentedColormap.from_list("feems_oc", ["#E8890C", "#FFFFFF", "#00C6D6"])

# symmetric about 0 so white always means "at the mean" -- do NOT let it auto-center
vmax = float(np.nanmax(np.abs(Z)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

fig = plt.figure(figsize=(7, 9), dpi=150)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([117.5, 127.0, 4.0, 21.5], crs=ccrs.PlateCarree())

pc = ax.pcolormesh(LON, LAT, Z, cmap=feems_cmap, norm=norm,
                   shading="auto", alpha=0.85, transform=ccrs.PlateCarree(), zorder=1)

ax.add_feature(cfeature.COASTLINE.with_scale("10m"), lw=0.5, edgecolor="black", zorder=5)

# sample locations
ax.scatter(coord[:, 0], coord[:, 1], s=18, c="black",
           transform=ccrs.PlateCarree(), zorder=10)

cb = fig.colorbar(pc, ax=ax, fraction=0.035, pad=0.03)
cb.set_label(r"$\log_{10}(w/\bar{w})$")

gl = ax.gridlines(draw_labels=True, lw=0.2, alpha=0.3)
gl.top_labels = False; gl.right_labels = False
plt.show()

In [ ]:
edge_arr = np.array(sp_graph.edges)          # 0-indexed, (n_edges, 2)

edgew_df = pd.DataFrame({
    "node_i": edge_arr[:, 0],
    "node_j": edge_arr[:, 1],
    "w": sp_graph.w,
    "log10_w_ratio": np.log10(sp_graph.w / sp_graph.w.mean()),
})
edgew_df.to_csv(UP/"edgew_grammar.csv", index=False)

nodepos_df = pd.DataFrame({
    "lon": sp_graph.node_pos[:, 0],
    "lat": sp_graph.node_pos[:, 1],
    "n_samples": [sp_graph.nodes[n]["n_samples"] for n in range(len(sp_graph.nodes))],
})
nodepos_df.to_csv(UP/"nodepos_grammar.csv", index=False)

print(edgew_df.shape, nodepos_df.shape)

In [ ]:
valid = ~np.ma.getmaskarray(Z)
raster_df = pd.DataFrame({
    "lon": LON[valid],
    "lat": LAT[valid],
    "log_w_ratio": Z[valid].data,
})
raster_df.to_csv(UP/"grammar_surface_raster.csv", index=False)
print(raster_df.shape)

In [ ]:
import json

meta = {
    "grid_res": GRID_RES,
    "habitat_source": "Natural Earth coastline (countries.geojson, name='Philippines')",
    "coast_buf": COAST_BUF,
    "n_nodes": int(grid.shape[0]),
    "n_edges": int(edges.shape[0]),
    "n_observed_nodes": int(sp_graph.n_observed_nodes),
    "scale_snps": bool(SCALE_SNPS),
    "use_idf": bool(USE_IDF),
    "optimize_q": None,
    "lamb_cv": float(lamb_cv),
    "n_features_retained": int(len(GB_KEEP)),
}
with open(UP / "grammar_feems_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(meta)